In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np
import forecast as F
from sklearn.ensemble import HistGradientBoostingRegressor

In [2]:
df = F.load_all(); X = F.make_features(df); y = df["demand"]
ok = X.notna().all(axis=1); X, y = X[ok], y[ok]
train = X.index < F.TEST_START; test = ~train
feats = F.CALENDAR + F.LAGS + F.WEATHER

In [3]:
def quantile_model(q):
    return HistGradientBoostingRegressor(
        loss="quantile", quantile=q, learning_rate=0.05, max_iter=1000,
        max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0,
        categorical_features=["day_of_week"], random_state=0,
    ).fit(X.loc[train, feats], y[train])

lo, hi = quantile_model(0.1), quantile_model(0.9)   # an 80% interval

def coverage(mask):
    inside = (y[mask] >= lo.predict(X.loc[mask, feats])) & (y[mask] <= hi.predict(X.loc[mask, feats]))
    return inside.mean()

print("coverage on training data:", coverage(train))
print("coverage on test year:   ", coverage(test))

coverage on training data: 0.7940989847715736
coverage on test year:    0.4841876926589793
